# Validation 4: Fair Comparison with node2vec
Compares GNN against node2vec + k-means on the same embedding space — a fair apples-to-apples comparison.

In [ ]:
!pip install torch torch_geometric node2vec scikit-learn pandas numpy matplotlib networkx

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.nn import Node2Vec
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import networkx as nx
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load data
with pd.HDFStore('./data/gene_network_data.h5') as store:
    tec = store['TEC']

np_tec_abs = np.abs(tec.to_numpy(copy=True))
gene_names = tec.index.tolist()
n_genes = len(gene_names)

threshold = 0.75
adj_matrix = (np_tec_abs > threshold).astype(float)
np.fill_diagonal(adj_matrix, 0)
edge_indices = np.where(adj_matrix > 0)
edge_index = torch.tensor(np.vstack([edge_indices[0], edge_indices[1]]), dtype=torch.long).to(device)
print(f'Loaded {n_genes} genes, {edge_index.shape[1]} edges')

In [ ]:
# Train GNN
class TEC_GNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_heads=8, dropout=0.3):
        super().__init__()
        self.gat1 = GATConv(input_dim, hidden_dim, heads=num_heads, dropout=dropout)
        self.gat2 = GATConv(hidden_dim * num_heads, hidden_dim, heads=1, dropout=dropout)
        self.dropout = dropout
    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.gat1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        embeddings = self.gat2(x, edge_index)
        return embeddings, torch.sigmoid(torch.mm(embeddings, embeddings.t()))

degrees = np.sum(adj_matrix, axis=1)
mean_corr = np.mean(np_tec_abs, axis=1)
x = torch.tensor(np.column_stack([degrees, mean_corr]), dtype=torch.float32).to(device)
target_adj = torch.tensor(adj_matrix, dtype=torch.float32).to(device)

gnn_model = TEC_GNN(input_dim=2).to(device)
optimizer = torch.optim.Adam(gnn_model.parameters(), lr=0.001)
gnn_model.train()
for epoch in range(200):
    optimizer.zero_grad()
    emb, recon = gnn_model(x, edge_index)
    loss = F.binary_cross_entropy(recon, target_adj)
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        print(f'GNN Epoch {epoch}: Loss={loss.item():.4f}')

gnn_model.eval()
with torch.no_grad():
    gnn_embeddings, _ = gnn_model(x, edge_index)
gnn_embeddings_np = gnn_embeddings.cpu().numpy()
print('GNN training done')

In [ ]:
# Train node2vec on same graph
print('Training node2vec...')
node2vec = Node2Vec(
    edge_index,
    embedding_dim=64,
    walk_length=20,
    context_size=10,
    walks_per_node=10,
    num_negative_samples=1,
    p=1.0,
    q=1.0,
    sparse=True
).to(device)

n2v_optimizer = torch.optim.SparseAdam(list(node2vec.parameters()), lr=0.01)
loader = node2vec.loader(batch_size=128, shuffle=True)

node2vec.train()
for epoch in range(50):
    total_loss = 0
    for pos_rw, neg_rw in loader:
        n2v_optimizer.zero_grad()
        loss = node2vec.loss(pos_rw.to(device), neg_rw.to(device))
        loss.backward()
        n2v_optimizer.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f'node2vec Epoch {epoch}: Loss={total_loss:.4f}')

node2vec.eval()
with torch.no_grad():
    n2v_embeddings_np = node2vec().cpu().numpy()
print('node2vec training done')

In [ ]:
# Compare clustering quality: GNN vs node2vec vs spectral
from sklearn.decomposition import PCA

results = {}
for name, emb in [('GNN (GAT)', gnn_embeddings_np), ('node2vec', n2v_embeddings_np)]:
    scores = []
    for k in range(2, 8):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(emb)
        s = silhouette_score(emb, labels)
        scores.append((k, s))
    best_k, best_s = max(scores, key=lambda x: x[1])
    results[name] = {'best_k': best_k, 'best_silhouette': best_s, 'all_scores': scores}
    print(f'{name}: best k={best_k}, silhouette={best_s:.4f}')

# Also add threshold-based baseline
from sklearn.preprocessing import normalize
adj_norm = normalize(adj_matrix, norm='l2')
scores_thresh = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(adj_norm)
    s = silhouette_score(adj_norm, labels)
    scores_thresh.append((k, s))
best_k_t, best_s_t = max(scores_thresh, key=lambda x: x[1])
results['Threshold (0.75)'] = {'best_k': best_k_t, 'best_silhouette': best_s_t}
print(f'Threshold baseline: best k={best_k_t}, silhouette={best_s_t:.4f}')

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

methods = list(results.keys())
sil_scores = [results[m]['best_silhouette'] for m in methods]
colors = ['#2196F3', '#FF9800', '#FF5722']

bars = axes[0].bar(methods, sil_scores, color=colors, width=0.5)
axes[0].set_ylabel('Best Silhouette Score', fontsize=12)
axes[0].set_title('Clustering Quality Comparison\n(Fair Embedding-Space Evaluation)', fontsize=12)
axes[0].set_ylim(-0.1, 1.1)
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
for bar, s in zip(bars, sil_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{s:.3f}', ha='center', fontsize=11, fontweight='bold')

# Silhouette vs k for GNN and node2vec
gnn_scores = results['GNN (GAT)']['all_scores']
n2v_scores = results['node2vec']['all_scores']
axes[1].plot([s[0] for s in gnn_scores], [s[1] for s in gnn_scores],
             'o-', color='#2196F3', lw=2, label='GNN (GAT)')
axes[1].plot([s[0] for s in n2v_scores], [s[1] for s in n2v_scores],
             's-', color='#FF9800', lw=2, label='node2vec')
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score vs Number of Clusters', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('node2vec_comparison_results.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: node2vec_comparison_results.png')

In [ ]:
with open('node2vec_comparison_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Results saved to node2vec_comparison_results.json')